# Forward Query Demo

Use PxFquery for perturbation-to-function questions. The result is a biological answer plus a reusable evidence object.

In [1]:
from pathlib import Path
import os
import sys
import warnings
from IPython.display import display

warnings.filterwarnings("ignore", message="IProgress not found.*")

repo_root = Path.cwd()
if not (repo_root / "src" / "pxfquery").exists() and (repo_root.parent / "src" / "pxfquery").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

# Optional: load local environment variables for the LLM provider.
for env_file in [Path.cwd() / ".env", Path.cwd().parent / ".env", Path.home() / ".env"]:
    if env_file.exists():
        for line in env_file.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ.setdefault(key.strip(), value.strip())

from pxfquery import PxFQuery

pxf = PxFQuery()
print("PxFquery", pxf.version)
resource_status = pxf.resources.status()
resource_info = resource_status.to_dict() if hasattr(resource_status, "to_dict") else dict(resource_status)
print("Resource status:")
print({
    "available": resource_info.get("available"),
    "source": resource_info.get("source"),
    "version": resource_info.get("version"),
    "available_file_count": len(resource_info.get("available_files") or {}),
})


PxFquery 0.5.12.dev0
Resource status:
{'available': True, 'source': 'manifest', 'version': 'v20260628', 'available_file_count': 18}


In [2]:
question = (
    "For EGFR-driven lung adenocarcinoma models, what functional programs are changed "
    "by EGFR inhibition, and are inflammatory or MAPK-related programs affected?"
)

print("Question:")
print(question)

qdata = pxf.tl.parse(question, top_n=10)
pxf.tl.answer(qdata)
answer = pxf.get.answer(qdata)

print("\nAnswer:")
print(answer)

Question:
For EGFR-driven lung adenocarcinoma models, what functional programs are changed by EGFR inhibition, and are inflammatory or MAPK-related programs affected?


[Parsing] start


[Parsing] done | mode=forward; context=EGFR-driven lung adenocarcinoma; perturbation=EGFR inhibition; time=2.82s
[Matching] start


[Matching] done | matches=6; time=4.93s
[Matrix] start


[Matrix] done | profiles=6; skipped=0; time=0.99s
[Evidence] start


[Evidence] done | status=ready; time=8.37s



Answer:
Answer
Overall: In EGFR-driven lung adenocarcinoma models, EGFR inhibition with gefitinib leads to increased stress, cholesterol homeostasis, interferon/MHC-II, and metabolism-related programs, and decreased cell cycle, MYC, and secreted programs. Inflammatory programs are supported as increased, while MAPK-related programs are not resolved from the supplied evidence.

Subquestion answers:
1. What functional programs are changed by EGFR inhibition? The main supported changes are increased stress, cholesterol homeostasis, interferon/MHC-II, bile acid metabolism, xenobiotic metabolism, adipogenesis, estrogen response, mTORC1 signaling, oxidative phosphorylation, fatty acid metabolism, heme metabolism, and cell cycle programs (E2F targets, G2/M checkpoint, MYC targets) in some models; decreased cell cycle (G1/S, G2/M), MYC, secreted proteins, respiration, spermatogenesis, and chromatin programs.
2. Are inflammatory programs affected? Supported. Interferon gamma response, interfer

In [3]:
print("Available answer tables:")
print(list(answer.tables.keys()))
print("\nRoute summary preview:")
display(answer.tables.get("route_summary", [])[:3])


Available answer tables:
['ranked_results', 'primary_route_ranked_results', 'route_summary', 'route_target_functions', 'route_function_results', 'matrix_context', 'claim_rules']

Route summary preview:


[{'route_id': 'forward_006',
  'status': 'routed',
  'cell': 'A549',
  'perturbation': 'BRD-K64052750',
  'perturbation_alias': 'gefitinib',
  'tier': 'proxy_cell_exact_perturbation',
  'cell_match_type': 'same_disease_cell',
  'perturbation_match_type': 'mechanism_representative',
  'cell_match_distance': 0.333,
  'perturbation_match_distance': 0.05,
  'route_quality_score': 0.192,
  'route_quality': 'strong_representative',
  'score_orientation': 'observed_drug_perturbation_effect',
  'recommended_operation': 'drug_treat',
  'reason': 'strict_pair_unavailable_or_additional_observed_evidence'},
 {'route_id': 'forward_007',
  'status': 'routed',
  'cell': 'BEN',
  'perturbation': 'BRD-K64052750',
  'perturbation_alias': 'gefitinib',
  'tier': 'proxy_cell_exact_perturbation',
  'cell_match_type': 'same_disease_cell',
  'perturbation_match_type': 'mechanism_representative',
  'cell_match_distance': 0.333,
  'perturbation_match_distance': 0.05,
  'route_quality_score': 0.192,
  'route_qua

In [4]:
print("Functional evidence preview:")
display(answer.tables.get("route_function_results", answer.biological_results)[:10])


Functional evidence preview:


[{'route_id': 'forward_006',
  'cell': 'A549',
  'perturbation': 'BRD-K64052750',
  'perturbation_alias': 'gefitinib',
  'modality': 'cp',
  'score_orientation': 'observed_drug_perturbation_effect',
  'recommended_operation': 'drug_treat',
  'route_quality_score': 0.192,
  'route_quality': 'strong_representative',
  'cell_match_type': 'same_disease_cell',
  'perturbation_match_type': 'mechanism_representative',
  'rank': 1,
  'label': 'HALLMARK_CHOLESTEROL_HOMEOSTASIS',
  'function': 'HALLMARK_CHOLESTEROL_HOMEOSTASIS',
  'score': 10.0,
  'direction': 'activated'},
 {'route_id': 'forward_006',
  'cell': 'A549',
  'perturbation': 'BRD-K64052750',
  'perturbation_alias': 'gefitinib',
  'modality': 'cp',
  'score_orientation': 'observed_drug_perturbation_effect',
  'recommended_operation': 'drug_treat',
  'route_quality_score': 0.192,
  'route_quality': 'strong_representative',
  'cell_match_type': 'same_disease_cell',
  'perturbation_match_type': 'mechanism_representative',
  'rank': 2,
 